# fst2framegraph One-Click Colab Run

Run this notebook top-to-bottom in a fresh runtime.


In [ ]:
import os
from pathlib import Path
import subprocess

# Deterministic environment guards for FrameSemanticTransformer in Colab
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

project_candidates = [
    Path('/content/fst2framegraph'),
    Path('/content/fst2framegraph-main'),
    Path('/content'),
]
project_dir = next((p for p in project_candidates if (p / 'pyproject.toml').exists()), None)
if project_dir is None:
    raise RuntimeError('Could not find project folder with pyproject.toml in /content. Upload/extract the repo first.')

%cd {project_dir}

subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.'], check=True)

print('Environment bootstrap complete.')


In [ ]:
from google.colab import files
from fst2framegraph import run_fst2graph

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload one CSV file to continue.')
csv_path = '/content/' + list(uploaded.keys())[0]

result = run_fst2graph(
    input_csv=csv_path,
    out_root='/content/fst2graph_runs',
    text_col='Transcript (text and audio)',
    id_col='Unique ID',
    doc_col='Unique ID',
    resume=True,
    batch_size=16,
    dedupe=True,
    random_seed=42,
)

print('Run complete:')
print(result['run_root'])
result


In [ ]:
import json
from pathlib import Path
import pandas as pd

run_root = Path(result['run_root'])
analysis_dir = Path(result['analysis_out_dir'])

print('Summary JSON:', run_root / 'summary.json')
print('Graph:', Path(result['graph_out_dir']) / 'graph.gpickle')
print('Top frames CSV:', analysis_dir / 'top_frame_types.csv')
print('Lift CSV:', analysis_dir / 'agent_frame_lift.csv')

display(pd.read_csv(analysis_dir / 'top_frame_types.csv').head(20))
display(pd.read_csv(analysis_dir / 'top_agent_fillers.csv').head(20))
display(pd.read_csv(analysis_dir / 'agent_frame_lift.csv').head(20))

summary = json.loads((run_root / 'summary.json').read_text(encoding='utf-8'))
summary
